In [8]:
import pandas as pd
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import tensorflow as tf
import matplotlib.image as mpimg
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
from sklearn.metrics import classification_report
from tensorflow.keras.losses import sparse_categorical_crossentropy
from tensorflow.keras.layers import Dense, Flatten, Conv2D, GlobalAveragePooling2D, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.metrics import Recall
from tensorflow.keras.models import Sequential
import segmentation_models as sm
from tensorflow.keras.metrics import BinaryIoU
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model

In [2]:
def load_split(base_dir, split):
    images_dir = os.path.join(base_dir, split, 'images')
    masks_dir  = os.path.join(base_dir, split, 'masks')
    
    X, y, labels = [], [], []
    
    # Iterate over numbered subfolders (1, 2, 3, ...)
    for subfolder in sorted(os.listdir(images_dir)):
        sub_img_dir  = os.path.join(images_dir, subfolder)
        sub_mask_dir = os.path.join(masks_dir,  subfolder)
        
        if not os.path.isdir(sub_img_dir):  # skip non-folders
            continue

        if subfolder == 'no_tumor':  # skip no_tumor subfolder for segmentation
            continue
        
        for fname in sorted(os.listdir(sub_img_dir)):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
                continue
            
            # Parse label: <global_id>_<patient_id>_<label>.ext
            label = fname.rsplit('.', 1)[0].split('_')[-1]
            
            img  = Image.open(os.path.join(sub_img_dir,  fname)).convert('RGB')
            mask = Image.open(os.path.join(sub_mask_dir, fname)).convert('L')
            
            img  = img.resize((224, 224))
            mask = mask.resize((224, 224))
            
            X.append(np.array(img))
            y.append(np.array(mask))
            labels.append(label)
    
    return np.array(X), np.array(y), labels

In [3]:
BASE_DIR = '../data/figshare_braintumor_split'
TRAIN_PATH = os.path.join(BASE_DIR, 'train', 'images', '1')
MASK_PATH = os.path.join(BASE_DIR, 'train', 'masks', '1')

In [4]:
X_train, y_train, train_labels = load_split(BASE_DIR, 'train')
X_val, y_val, val_labels = load_split(BASE_DIR, 'val')
X_test, y_test, test_labels = load_split(BASE_DIR, 'test')

# Normalize pixel values to [0, 1]
X_train = X_train / 255.0
X_val = X_val / 255.0
X_test = X_test / 255.0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2212, 224, 224, 3)
y_train shape: (2212, 224, 224)
X_val shape: (222, 224, 224, 3)
y_val shape: (222, 224, 224)
X_test shape: (630, 224, 224, 3)
y_test shape: (630, 224, 224)


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(train_labels)
y_train_enc = le.transform(train_labels)
y_val_enc = le.transform(val_labels)
y_test_enc = le.transform(test_labels)

print("Classes:", le.classes_)

Classes: ['1' '2' '3']


In [5]:
import numpy as np

# Binarize masks + add channel dimension
y_train_mask = np.expand_dims((y_train > 127).astype(np.float32), axis=-1)
y_val_mask   = np.expand_dims((y_val > 127).astype(np.float32), axis=-1)
y_test_mask  = np.expand_dims((y_test > 127).astype(np.float32), axis=-1)

print("New mask shapes:")
print("Train:", y_train_mask.shape)
print("Val:", y_val_mask.shape)
print("Test:", y_test_mask.shape)

New mask shapes:
Train: (2212, 224, 224, 1)
Val: (222, 224, 224, 1)
Test: (630, 224, 224, 1)


In [6]:
import keras
iou_metric = keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou')

In [24]:
# LOAD MODELS

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# Load U-Net (Segmentation)
unet = load_model(
    '../models/brain_tumor_unet.keras', 
    custom_objects={
        'dice_loss': sm.losses.DiceLoss(), 
        'dice_coef': dice_coef,
        'iou': BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou')
    }
)

In [21]:
unet.evaluate(X_test, y_test_mask)

20/20 [==============================] - 5s 233ms/step - loss: 0.2933 - dice_coef: 0.7077 - iou: 0.5438 - accuracy: 0.9902


[0.2932955026626587,
 0.7077425122261047,
 0.5437626838684082,
 0.9901633858680725]

In [25]:
from sklearn.metrics import  f1_score
y_pred = unet.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(np.float32)

y_true_flat = y_test_mask.flatten()
y_pred_flat = y_pred_binary.flatten()

print(f"Dice Score:  {f1_score(y_true_flat, y_pred_flat):.4f}")

20/20 [==============================] - 4s 222ms/step
Dice Score:  0.7045
